# Paper Analyses — ENVSOFT-D-26-00848
**Monitoring Reservoir Surface and Storage Dynamics Using Sentinel-1 SAR and ML in GEE**

Este notebook executa as análises prioritárias para a revisão do artigo:

| # | Análise | Saída |
|---|---------|-------|
| 1 | **Acurácia SVM (split 70/30)** — VV+VH vs VV-only | Print no console |
| 2 | **A/P para os 41 reservatórios** | CSV + figura |
| 3 | **Ablação VH + baseline Otsu** — séries temporais de área | 3 CSVs por reservatório no Drive |
| 4 | **JRC auto-training** — comparação com treinamento manual | Print no console |

## Pré-requisitos
```bash
pip install earthengine-api geemap pandas matplotlib
earthengine authenticate   # só uma vez — abre o browser
```

Substitua `'your-gee-project'` na célula de autenticação pelo ID do projeto GEE da UNIPA.

In [ ]:
# Verificação de pacotes
import subprocess, sys
for pkg in ['ee', 'pandas', 'matplotlib']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                               {'ee': 'earthengine-api', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}[pkg]])

import ee
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

print('Imports OK')

In [ ]:
# ── Autenticação ──────────────────────────────────────────────────────────────
# Na primeira execução descomente ee.Authenticate() e siga o browser.
# Nas execuções seguintes só Initialize() é necessário.

GEE_PROJECT = 'your-gee-project'   # ← substituir pelo ID do projeto UNIPA

# ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print(f'Earth Engine inicializado — projeto: {GEE_PROJECT}')

## Parser de `entries.js`

Lê os polígonos AOI (41 reservatórios), a região de treinamento (`roi_training`) e
os polígonos de amostras (`agua_training`, `terra_training`) diretamente do arquivo JS,
sem necessidade de exportar assets adicionais do Code Editor.

In [ ]:
def _extract_brackets(text, start):
    """Retorna a substring com colchetes balanceados a partir de `start`."""
    depth = 0
    for i, c in enumerate(text[start:], start):
        if c == '[':
            depth += 1
        elif c == ']':
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def _extract_parens(text, start):
    """Retorna conteúdo entre parênteses balanceados a partir de `start`."""
    depth = 0
    for i, c in enumerate(text[start:], start):
        if c == '(':
            depth += 1
        elif c == ')':
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def parse_entries_js(filepath):
    """
    Faz parse de entries.js e retorna:
      - aoi        : dict {nome_reservatorio: ee.Geometry}
      - roi_training  : ee.Geometry.MultiPolygon
      - agua_training : ee.FeatureCollection  (landcover=1)
      - terra_training: ee.FeatureCollection  (landcover=2)
    """
    text = Path(filepath).read_text(encoding='utf-8')

    # Mapeamento: variável JS → nome do reservatório (de m.AOI em reservoirs_s1_svm.js)
    AOI_VAR_MAP = {
        'Ancipa'        : 'Invaso Ancipa',
        'Arancio'       : 'Invaso Arancio',
        'Castello'      : 'Invaso Castello',
        'Cimia'         : 'Invaso Cimia',
        'Comunelli'     : 'Invaso Comunelli',
        'DiGela'        : 'Invaso di Gela',
        'Gammauta'      : 'Invaso di Gammauta',
        'Albanesi'      : 'Invaso di Piana degli Albanesi',
        'Diddino'       : 'Invaso Diddino',
        'Dirillo'       : 'Invaso Dirillo',
        'Disueri'       : 'Invaso Disueri',
        'DonSturzo'     : 'Invaso Don Sturzo',
        'Fanaco'        : 'Invaso Fanaco',
        'FiumaraGrande' : 'Invaso Fiumara Grande',
        'Furore'        : 'Invaso Furore',
        'Garcia'        : 'Invaso Garcia',
        'Gibbesi'       : 'Invaso Gibbesi',
        'Gorgo'         : 'Invaso Gorgo',
        'Guadalami'     : 'Invaso Guadalami',
        'Lentini'       : 'Invaso Lentini',
        'MonteCavallaro': 'Invaso Monte Cavallaro',
        'Nicoletti'     : 'Invaso Nicoletti',
        'Olivo'         : 'Invaso Olivo',
        'Paceco'        : 'Invaso Paceco',
        'Pergusa'       : 'Invaso Pergusa',
        'Leone'         : 'Invaso Piano del Leone',
        'Pietrarossa'   : 'Invaso Pietrarossa',
        'Poma'          : 'Invaso Poma',
        'PonteBarca'    : 'Invaso Ponte Barca',
        'Pozzillo'      : 'Invaso Pozzillo',
        'Prizzi'        : 'Invaso Prizzi',
        'Rosamarina'    : 'Invaso Rosamarina',
        'Rubino'        : 'Invaso Rubino',
        'SanGiovanni'   : 'Invaso San Giovanni',
        'SantaRosalia'  : 'Invaso Santa Rosalia',
        'Scanzano'      : 'Invaso Scanzano',
        'Sciaguana'     : 'Invaso Sciaguana',
        'Trinita'       : 'Invaso Trinità',
        'VascaOgliastro': 'Invaso Vasca Ogliastro',
        'Villarosa'     : 'Invaso Villarosa',
        'Zaffarana'     : 'Invaso Zaffarana',
    }

    def parse_geometry(var_name):
        """Extrai ee.Geometry de uma variável (Polygon ou MultiPolygon)."""
        for geom_type in ('Polygon', 'MultiPolygon'):
            pat = rf'(?:var\s+)?\b{re.escape(var_name)}\b\s*=\s*(?:/\*[^*]*\*/\s*)*ee\.Geometry\.{geom_type}\s*\('
            m = re.search(pat, text)
            if not m:
                continue
            pos = m.end()
            while pos < len(text) and text[pos] != '[':
                pos += 1
            coords_str = _extract_brackets(text, pos)
            if not coords_str:
                continue
            try:
                coords = json.loads(coords_str)
                if geom_type == 'Polygon':
                    return ee.Geometry.Polygon(coords[0])
                else:
                    return ee.Geometry.MultiPolygon(coords)
            except Exception as e:
                print(f'  Aviso: {var_name} ({geom_type}): {e}')
        return None

    # AOIs
    aoi = {}
    for var, name in AOI_VAR_MAP.items():
        g = parse_geometry(var)
        if g:
            aoi[name] = g

    # roi_training
    roi_training = parse_geometry('roi_training')

    # agua_training e terra_training (FeatureCollections de polígonos)
    def parse_fc(var_name, landcover):
        pat = rf'\b{re.escape(var_name)}\b\s*=\s*(?:/\*[^*]*\*/\s*)*ee\.FeatureCollection\s*\('
        m = re.search(pat, text)
        if not m:
            return None
        fc_body = _extract_parens(text, m.end() - 1)  # inclui os parênteses
        if not fc_body:
            return None
        features = []
        for pm in re.finditer(r'ee\.Geometry\.Polygon\s*\(', fc_body):
            p = pm.end()
            while p < len(fc_body) and fc_body[p] != '[':
                p += 1
            cstr = _extract_brackets(fc_body, p)
            if cstr:
                try:
                    c = json.loads(cstr)
                    features.append(ee.Feature(ee.Geometry.Polygon(c[0]), {'landcover': landcover}))
                except Exception:
                    pass
        return ee.FeatureCollection(features) if features else None

    agua_training = parse_fc('agua_training', 1)
    terra_training = parse_fc('terra_training', 2)

    return {
        'aoi': aoi,
        'roi_training': roi_training,
        'agua_training': agua_training,
        'terra_training': terra_training,
    }


# Caminho relativo ao notebook (analysis/ → main_script/)
ENTRIES_PATH = Path('../main_script/entries.js')
parsed = parse_entries_js(ENTRIES_PATH)

aoi_dict       = parsed['aoi']
roi_training   = parsed['roi_training']
agua_training  = parsed['agua_training']
terra_training = parsed['terra_training']

print(f'AOIs carregados:        {len(aoi_dict)} reservatórios')
print(f'Polígonos de água:      {agua_training.size().getInfo()}')
print(f'Polígonos de terra:     {terra_training.size().getInfo()}')
print(f'roi_training carregado: {roi_training is not None}')

In [ ]:
# ── Constantes ────────────────────────────────────────────────────────────────
BANDS            = ['VV', 'VH']
SMOOTHING_RADIUS = 30
TRAINING_YEAR    = 2023
VALIDATION_START = '2024-05-01'
VALIDATION_END   = '2025-05-31'

VALIDATION_RESERVOIRS = [
    'Invaso Rosamarina',
    'Invaso Pozzillo',
    'Invaso Poma',
    'Invaso Ancipa',
]

# Coeficientes AEV: V = a·A² + b·A + c  (A em ha, V em 10⁶ m³)
LAKE_COEFFICIENTS = {
    'Invaso Rosamarina': {'a':  0.0003, 'b':  0.0817, 'c': -0.5108},
    'Invaso Pozzillo':   {'a':  0.0003, 'b': -0.0610, 'c':  2.7448},
    'Invaso Poma':       {'a':  0.0001, 'b':  0.0817, 'c': -2.1140},
    'Invaso Ancipa':     {'a':  0.0009, 'b':  0.0938, 'c': -2.0159},
}

OUTPUT_DIR = Path('../validation_data/statistics/area_statistics')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Constantes definidas.')

## Preparar dados de treinamento

Replica a lógica da seção `SENTINEL-1 TRAINING` de `reservoirs_s1_svm.js`:
mosaico S1 de 2023 sobre as 10 regiões de treinamento sicilianas → amostragem dos
polígonos manuais → limpeza → `trainingClean`.

In [ ]:
def prepare_training_data(roi_training, agua_fc, terra_fc, year=TRAINING_YEAR):
    s1_mosaic = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filterMetadata('resolution_meters', 'equals', 10)
        .filterBounds(roi_training)
        .select('VV', 'VH')
        .filterDate(f'{year}-01-01', f'{year + 1}-01-01')
        .mosaic()
        .focal_mean(SMOOTHING_RADIUS, 'circle', 'meters')
        .clip(roi_training)
    )

    all_polygons = agua_fc.merge(terra_fc)

    training = s1_mosaic.select(BANDS).sampleRegions(
        collection=all_polygons,
        properties=['landcover'],
        scale=30
    )

    training_clean = (
        training
        .filter(ee.Filter.inList('landcover', [1, 2]))
        .filter(ee.Filter.notNull(['landcover']))
    )
    return training_clean


print('Amostrando dados de treinamento (pode levar 1–2 min)...')
training_clean = prepare_training_data(roi_training, agua_training, terra_training)
n_total = training_clean.size().getInfo()
n_water = training_clean.filter(ee.Filter.eq('landcover', 1)).size().getInfo()
n_land  = training_clean.filter(ee.Filter.eq('landcover', 2)).size().getInfo()
print(f'Total de amostras: {n_total}  (água={n_water}, terra={n_land})')

## Análise 1 — Acurácia SVM com split 70/30

Treina o SVM em 70% das amostras e avalia no 30% restante, para VV+VH e VV-only.
Produz os valores a reportar na tabela de acurácia do artigo.

In [ ]:
def accuracy_split(training_clean, seed=42):
    """Retorna dict com OA, Kappa e matriz de confusão para VV+VH e VV-only."""
    with_random = training_clean.randomColumn('random', seed)
    train_set   = with_random.filter(ee.Filter.lte('random', 0.7))
    test_set    = with_random.filter(ee.Filter.gt('random', 0.7))

    results = {}
    for label, input_bands in [('VV+VH', ['VV', 'VH']), ('VV-only', ['VV'])]:
        clf = (
            ee.Classifier.libsvm(kernelType='RBF', cost=1, gamma=0.01)
            .train(features=train_set, classProperty='landcover', inputProperties=input_bands)
        )
        em = test_set.classify(clf).errorMatrix('landcover', 'classification')
        results[label] = {
            'OA'    : em.accuracy().getInfo(),
            'Kappa' : em.kappa().getInfo(),
            'matrix': em.getInfo(),
        }

    n_train = train_set.size().getInfo()
    n_test  = test_set.size().getInfo()
    return results, n_train, n_test


print('Calculando acurácia (split 70/30, seed=42)...')
accuracy_results, n_train, n_test = accuracy_split(training_clean)

print(f'\nN treino: {n_train}   N teste: {n_test}')
print()
for label, res in accuracy_results.items():
    print(f'{label:10s}  OA={res["OA"]:.4f}   Kappa={res["Kappa"]:.4f}')
    print(f'  Matriz de confusão (linhas=real, colunas=predito):')
    mat = res['matrix']
    df_cm = pd.DataFrame(
        mat['array'],
        index  =[f'Real {v}' for v in mat['axis_labels'][0]],
        columns=[f'Pred {v}' for v in mat['axis_labels'][1]]
    )
    print(df_cm, '\n')

## Análise 2 — Razão A/P para os 41 reservatórios

Computa Área, Perímetro e razão A/P a partir dos polígonos AOI de `entries.js`.
Exporta CSV para o Drive e gera figura para o artigo.

In [ ]:
def compute_morphometrics(aoi_dict):
    """Computa A/P para todos os reservatórios e retorna FeatureCollection + DataFrame."""
    features = []
    for name, geom in aoi_dict.items():
        area  = geom.area(maxError=1)
        perim = geom.perimeter(maxError=1)
        features.append(ee.Feature(geom.centroid(1), {
            'name'    : name,
            'area_m2' : area,
            'perim_m' : perim,
            'AP_ratio': area.divide(perim),
        }))

    fc = ee.FeatureCollection(features)

    # Exportar para o Drive (aparece no painel Tasks do GEE)
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description='reservoir_AP_all41',
        fileFormat='CSV',
        selectors=['name', 'area_m2', 'perim_m', 'AP_ratio']
    )
    task.start()
    print(f'Export task ID: {task.status()["id"]}')

    # Buscar resultados imediatamente para visualização local
    data = fc.getInfo()
    rows = [f['properties'] for f in data['features']]
    df = pd.DataFrame(rows).sort_values('AP_ratio', ascending=False).reset_index(drop=True)
    return fc, df


print('Calculando morfometria dos 41 reservatórios...')
fc_morph, df_morph = compute_morphometrics(aoi_dict)
print(f'\nTop 5 mais compactos (maior A/P):')
print(df_morph[['name', 'area_m2', 'perim_m', 'AP_ratio']].head())
print(f'\nTop 5 menos compactos (menor A/P):')
print(df_morph[['name', 'area_m2', 'perim_m', 'AP_ratio']].tail())

In [ ]:
# ── Figura: distribuição A/P dos 41 reservatórios ─────────────────────────────

VAL_SHORT = {n: n.replace('Invaso ', '') for n in VALIDATION_RESERVOIRS}
COLORS = {
    'Invaso Rosamarina': '#e41a1c',
    'Invaso Pozzillo':   '#ff7f00',
    'Invaso Poma':       '#4daf4a',
    'Invaso Ancipa':     '#984ea3',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico de barras horizontal
ax = axes[0]
bar_colors = [COLORS.get(n, '#aec7e8') for n in df_morph['name']]
labels = df_morph['name'].str.replace('Invaso ', '', regex=False)
bars = ax.barh(labels, df_morph['AP_ratio'], color=bar_colors, edgecolor='none', height=0.7)
ax.set_xlabel('A/P Ratio (m)', fontsize=11)
ax.set_title('Shoreline Compactness — All 41 Reservoirs', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.tick_params(axis='y', labelsize=7)

# Legenda dos reservatórios de validação
patches = [mpatches.Patch(color=c, label=VAL_SHORT[n]) for n, c in COLORS.items()]
ax.legend(handles=patches, title='Validation\nreservoirs', fontsize=8, loc='lower right')

# Histograma da distribuição
ax2 = axes[1]
ax2.hist(df_morph['AP_ratio'], bins=12, color='#aec7e8', edgecolor='white', zorder=2)
ax2.set_xlabel('A/P Ratio (m)', fontsize=11)
ax2.set_ylabel('Number of reservoirs', fontsize=11)
ax2.set_title('Distribution of A/P Ratios', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.4, zorder=1)

for name, color in COLORS.items():
    val = df_morph.loc[df_morph['name'] == name, 'AP_ratio']
    if not val.empty:
        ax2.axvline(val.values[0], color=color, lw=2,
                    label=VAL_SHORT[name], zorder=3)
ax2.legend(fontsize=9)

plt.tight_layout()
fig_path = Path('../validation_data/morphometric_analysis/AP_all41_distribution.png')
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva em: {fig_path}')

## Análise 3 — Ablação VH + Baseline Otsu

Para cada um dos 4 reservatórios de validação e o período mai/2024–mai/2025, exporta
três séries temporais de área (ha) para o Google Drive:

1. **SVM VV+VH** — classificador principal
2. **SVM VV-only** — ablação do canal VH
3. **Otsu VV** — baseline não-supervisionado

Os CSVs são depois lidos pelo script MATLAB de estatísticas para gerar a Tabela 3.

In [ ]:
# Treinar classificadores no dataset completo
classifier_vvvh = (
    ee.Classifier.libsvm(kernelType='RBF', cost=1, gamma=0.01)
    .train(features=training_clean, classProperty='landcover', inputProperties=['VV', 'VH'])
)
classifier_vv = (
    ee.Classifier.libsvm(kernelType='RBF', cost=1, gamma=0.01)
    .train(features=training_clean, classProperty='landcover', inputProperties=['VV'])
)


def get_s1_collection(aoi, start, end):
    return (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterDate(start, end)
        .filterBounds(aoi)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.eq('resolution_meters', 10))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select('VV', 'VH')
    )


def otsu_water(image, aoi):
    histogram = ee.Dictionary(
        image.select('VV').reduceRegion(
            reducer=ee.Reducer.histogram(255, 2),
            geometry=aoi, scale=10, maxPixels=1e9
        ).get('VV')
    )
    threshold = ee.Algorithms.Otsu(histogram)
    return image.select('VV').lt(threshold).rename('WaterOtsu')


def classify_svm(image, aoi, clf, input_bands, out_name):
    filtered = image.clip(aoi).focal_mean(SMOOTHING_RADIUS, 'circle', 'meters')
    return filtered.select(input_bands).classify(clf).eq(1).rename(out_name)


def area_feature(image, mask_band, area_prop, aoi):
    area_m2 = (
        image.select(mask_band).unmask(0).clip(aoi)
        .multiply(ee.Image.pixelArea())
        .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi, scale=10, maxPixels=1e9)
        .get(mask_band)
    )
    return ee.Feature(None, {
        'data': image.date().format('YYYY-MM-dd'),
        'system:time_start': image.date().millis(),
        area_prop: ee.Number(area_m2).divide(10000),
    })


def export_ablation(reservoir_name, aoi, start=VALIDATION_START, end=VALIDATION_END):
    safe  = reservoir_name.replace(' ', '_').replace('/', '_')
    coll  = get_s1_collection(aoi, start, end)

    ts_svm = coll.map(lambda img: area_feature(
        img.addBands(classify_svm(img, aoi, classifier_vvvh, ['VV', 'VH'], 'WaterSVM')),
        'WaterSVM', 'areaSVM_ha', aoi))

    ts_vv = coll.map(lambda img: area_feature(
        img.addBands(classify_svm(img, aoi, classifier_vv, ['VV'], 'WaterVV')),
        'WaterVV', 'areaVVonly_ha', aoi))

    ts_otsu = coll.map(lambda img: area_feature(
        img.addBands(otsu_water(img, aoi)),
        'WaterOtsu', 'areaOtsu_ha', aoi))

    tasks_started = []
    for ts, desc, sel in [
        (ts_svm,  f'area_SVM_VVpVH_{safe}',  ['data', 'areaSVM_ha']),
        (ts_vv,   f'area_SVM_VVonly_{safe}',  ['data', 'areaVVonly_ha']),
        (ts_otsu, f'area_Otsu_{safe}',        ['data', 'areaOtsu_ha']),
    ]:
        t = ee.batch.Export.table.toDrive(
            collection=ts, description=desc, fileFormat='CSV', selectors=sel)
        t.start()
        tasks_started.append(t.status()['id'])
        print(f'  Queued: {desc}')

    return tasks_started


print('Iniciando exports de ablação para os 4 reservatórios de validação...')
print(f'Período: {VALIDATION_START} → {VALIDATION_END}\n')
all_tasks = []
for name in VALIDATION_RESERVOIRS:
    print(f'{name}:')
    ids = export_ablation(name, aoi_dict[name])
    all_tasks.extend(ids)

print(f'\n{len(all_tasks)} tasks iniciadas.')
print('Monitore em: https://code.earthengine.google.com/tasks')

## Análise 4 — JRC Auto-Training (Escalabilidade)

Gera amostras de treinamento automaticamente a partir do JRC Global Surface Water
(ocorrência ≥95% = água permanente; =0% no buffer de 500 m = terra) e treina o SVM
para cada reservatório de validação. Compara acurácia com o treinamento manual.

In [ ]:
def auto_generate_jrc_samples(aoi, n_per_class=500, seed=42):
    """Gera amostras de treinamento automaticamente via JRC Global Surface Water."""
    jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')

    # Água: pixels com presença ≥95% dentro do AOI
    water_s = (
        jrc.gte(95).selfMask().clip(aoi)
        .sample(region=aoi, scale=30, numPixels=n_per_class, seed=seed, geometries=True)
        .map(lambda f: f.set('landcover', 1))
    )

    # Terra: pixels com presença =0% no buffer externo de 500 m
    buffer = aoi.buffer(500).difference(aoi)
    land_s = (
        jrc.eq(0).selfMask().clip(buffer)
        .sample(region=buffer, scale=30, numPixels=n_per_class, seed=seed, geometries=True)
        .map(lambda f: f.set('landcover', 2))
    )

    return water_s.merge(land_s)


def train_from_jrc(name, aoi, year=2023, seed=42):
    """Treina SVM com amostras JRC para um reservatório e retorna métricas."""
    jrc_samples = auto_generate_jrc_samples(aoi)

    s1 = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filterMetadata('resolution_meters', 'equals', 10)
        .filterBounds(aoi)
        .filterDate(f'{year}-01-01', f'{year + 1}-01-01')
        .select('VV', 'VH')
        .mosaic()
        .focal_mean(SMOOTHING_RADIUS, 'circle', 'meters')
        .clip(aoi)
    )

    training_jrc = (
        s1.select(['VV', 'VH']).sampleRegions(
            collection=jrc_samples, properties=['landcover'], scale=30
        ).filter(ee.Filter.notNull(['landcover', 'VV', 'VH']))
    )

    split    = training_jrc.randomColumn('random', seed)
    train_j  = split.filter(ee.Filter.lte('random', 0.7))
    test_j   = split.filter(ee.Filter.gt('random', 0.7))

    clf_jrc = (
        ee.Classifier.libsvm(kernelType='RBF', cost=1, gamma=0.01)
        .train(features=train_j, classProperty='landcover', inputProperties=['VV', 'VH'])
    )

    em  = test_j.classify(clf_jrc).errorMatrix('landcover', 'classification')
    oa  = em.accuracy().getInfo()
    kap = em.kappa().getInfo()
    n_w = jrc_samples.filter(ee.Filter.eq('landcover', 1)).size().getInfo()
    n_l = jrc_samples.filter(ee.Filter.eq('landcover', 2)).size().getInfo()

    return {'OA': oa, 'Kappa': kap, 'n_water': n_w, 'n_land': n_l, 'classifier': clf_jrc}


print('=== JRC Auto-Training por reservatório ===\n')
print(f'Referência — Manual SVM VV+VH (pool 10 regiões sicilianas):')
print(f'  OA={accuracy_results["VV+VH"]["OA"]:.4f}  Kappa={accuracy_results["VV+VH"]["Kappa"]:.4f}\n')

jrc_results = {}
for name in VALIDATION_RESERVOIRS:
    short = name.replace('Invaso ', '')
    print(f'{short}: ', end='', flush=True)
    res = train_from_jrc(name, aoi_dict[name])
    jrc_results[name] = res
    print(f'OA={res["OA"]:.4f}  Kappa={res["Kappa"]:.4f}  '
          f'(água JRC={res["n_water"]}, terra JRC={res["n_land"]})')

## Tabela-resumo para o artigo

Consolida os resultados das análises 1–4 em uma tabela comparativa.

In [ ]:
rows = []
for name in VALIDATION_RESERVOIRS:
    short = name.replace('Invaso ', '')
    ap_val = df_morph.loc[df_morph['name'] == name, 'AP_ratio']
    jrc = jrc_results.get(name, {})
    rows.append({
        'Reservoir'                  : short,
        'A/P (m)'                    : round(ap_val.values[0], 1) if not ap_val.empty else '-',
        'Manual SVM OA (VV+VH test)' : f"{accuracy_results['VV+VH']['OA']:.4f}",
        'Manual SVM OA (VV-only test)': f"{accuracy_results['VV-only']['OA']:.4f}",
        'JRC-trained SVM OA'         : f"{jrc.get('OA', '-'):.4f}" if jrc.get('OA') else '-',
        'JRC Kappa'                  : f"{jrc.get('Kappa', '-'):.4f}" if jrc.get('Kappa') else '-',
    })

df_summary = pd.DataFrame(rows)
print('=== Tabela-resumo das análises ===\n')
print(df_summary.to_string(index=False))

csv_path = OUTPUT_DIR / 'summary_accuracy_comparison.csv'
df_summary.to_csv(csv_path, index=False)
print(f'\nCSV salvo em: {csv_path}')

In [ ]:
# ── Figura: comparação de acurácia manual vs JRC por reservatório ─────────────
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(VALIDATION_RESERVOIRS))
w = 0.25
shorts = [n.replace('Invaso ', '') for n in VALIDATION_RESERVOIRS]

oa_vvvh  = [accuracy_results['VV+VH']['OA']]  * len(VALIDATION_RESERVOIRS)
oa_vvonly= [accuracy_results['VV-only']['OA']] * len(VALIDATION_RESERVOIRS)
oa_jrc   = [jrc_results[n]['OA'] for n in VALIDATION_RESERVOIRS]

ax.bar(x - w, oa_vvvh,   width=w, label='Manual SVM VV+VH',  color='#2166ac')
ax.bar(x,     oa_vvonly,  width=w, label='Manual SVM VV-only', color='#92c5de')
ax.bar(x + w, oa_jrc,     width=w, label='JRC auto-training',  color='#f4a582')

ax.set_xticks(x)
ax.set_xticklabels(shorts, fontsize=10)
ax.set_ylim(0.85, 1.01)
ax.set_ylabel('Overall Accuracy (test set)', fontsize=11)
ax.set_title('SVM Accuracy Comparison: Manual vs JRC Auto-Training', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
fig_path2 = Path('../validation_data/morphometric_analysis/accuracy_comparison.png')
plt.savefig(fig_path2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva em: {fig_path2}')

---
## Análise 5 — Comparação quantitativa vs. PlanetScope (Tabela 3)

Lê os dados locais já exportados do GEE (ablação) e extraídos manualmente do app,
aplica a mesma cadeia de pós-processamento do pipeline GEE e calcula:

- **RMSE** (Root Mean Square Error) em hectares
- **Bias** médio (método − Planet)
- **KGE** (Kling-Gupta Efficiency)

para cada combinação método × reservatório.

In [ ]:
import csv, math
from datetime import datetime, timedelta
from collections import OrderedDict
from pathlib import Path

BASE_ABL  = Path('../validation_data/GEEvalidation-extracted')
BASE_STAT = Path('../validation_data/statistics/area_statistics')
LAKES = ['Rosamarina', 'Pozzillo', 'Poma', 'Ancipa']

SVM_FILES    = {l: BASE_STAT / f'{l.lower()}SVM.csv'    for l in LAKES}
PLANET_FILES = {l: BASE_STAT / f'{l.lower()}Planet.csv' for l in LAKES}

DATE_FMTS = ['%Y-%m-%d', '%d-%b-%y', '%d-%b-%Y', '%d/%m/%Y']

def parse_date(s):
    for fmt in DATE_FMTS:
        try: return datetime.strptime(s.strip(), fmt)
        except: pass
    raise ValueError(f'Cannot parse date: {s!r}')

def read_csv_dedup(path, date_col='data', val_col='area'):
    seen = OrderedDict()
    with open(path, encoding='utf-8-sig') as f:
        for r in csv.DictReader(f):
            d = parse_date(r[date_col])
            if d not in seen:
                seen[d] = float(r[val_col])
    return list(seen.keys()), list(seen.values())

def read_ablation_raw(lake, method, val_col):
    path = BASE_ABL / f'area_{method}_Invaso_{lake}.csv'
    seen = OrderedDict()
    with open(path, encoding='utf-8-sig') as f:
        for r in csv.DictReader(f):
            d = r['data']
            if d not in seen:
                seen[d] = float(r[val_col])
    dates = [datetime.strptime(k, '%Y-%m-%d') for k in seen]
    return dates, list(seen.values())

print('I/O helpers OK')


In [ ]:
def remove_outliers_global(dates, areas, thr=2.0):
    m = sum(areas) / len(areas)
    s = math.sqrt(sum((a - m)**2 for a in areas) / len(areas))
    pairs = [(d, a) for d, a in zip(dates, areas) if s == 0 or abs(a - m)/s <= thr]
    return [p[0] for p in pairs], [p[1] for p in pairs]

def remove_outliers_local(dates, areas, window=5, sigma=1.5):
    n, hw = len(areas), window // 2
    out = []
    for i in range(n):
        win = areas[max(0, i-hw):min(n, i+hw+1)]
        m = sum(win) / len(win)
        s = math.sqrt(sum((v-m)**2 for v in win)/len(win)) if len(win) > 1 else 0
        if s == 0 or abs(areas[i]-m)/s <= sigma:
            out.append((dates[i], areas[i]))
    return [x[0] for x in out], [x[1] for x in out]

def lowess_smooth(dates, areas, wd=20, bw=7):
    out = []
    for cd, ca in zip(dates, areas):
        ws, we = cd - timedelta(days=wd), cd + timedelta(days=wd)
        wlist = [(math.exp(-((abs((cd-d).days)/bw)**2)), a)
                 for d, a in zip(dates, areas) if ws <= d <= we]
        sw = sum(w for w, _ in wlist)
        out.append(sum(w*a for w, a in wlist)/sw if sw else ca)
    return out

def apply_pipeline(dates, areas):
    d, a = remove_outliers_global(dates, areas)
    d, a = remove_outliers_local(d, a, 5,  1.5)
    d, a = remove_outliers_local(d, a, 5,  1.5)
    d, a = remove_outliers_local(d, a, 10, 1.5)
    return d, lowess_smooth(d, a)

print('Pipeline OK')


In [ ]:
def kge(obs, sim):
    n = len(obs)
    if n < 2: return float('nan')
    mo, ms   = sum(obs)/n, sum(sim)/n
    so       = math.sqrt(sum((v-mo)**2 for v in obs)/n)
    ss       = math.sqrt(sum((v-ms)**2 for v in sim)/n)
    r_num    = sum((o-mo)*(s-ms) for o, s in zip(obs, sim))
    r        = r_num / (n*so*ss) if so*ss != 0 else 0
    beta     = ms/mo if mo != 0 else float('nan')
    gamma    = (ss/ms)/(so/mo) if mo != 0 and ms != 0 else float('nan')
    return 1 - math.sqrt((r-1)**2 + (beta-1)**2 + (gamma-1)**2)

def match_nearest(ref_d, ref_v, tgt_d, tgt_v, max_days=6):
    tgt = dict(zip(tgt_d, tgt_v))
    obs, sim = [], []
    for rd, rv in zip(ref_d, ref_v):
        cands = [(abs((rd-d).days), v) for d, v in tgt.items()
                 if abs((rd-d).days) <= max_days]
        if cands:
            obs.append(rv)
            sim.append(min(cands)[1])
    return obs, sim

METHODS = [
    ('SVM VV+VH (app)', 'app_svm',  None,          None),
    ('SVM VV-only',     'ablation', 'SVM_VVonly',  'areaVVonly_ha'),
    ('Otsu VV',         'ablation', 'Otsu',        'areaOtsu_ha'),
]

records = []
for lake in LAKES:
    pl_d, pl_a = read_csv_dedup(PLANET_FILES[lake])
    for label, source, raw_method, raw_col in METHODS:
        if source == 'app_svm':
            sim_d, sim_a = read_csv_dedup(SVM_FILES[lake])
        else:
            rd, ra = read_ablation_raw(lake, raw_method, raw_col)
            sim_d, sim_a = apply_pipeline(rd, ra)
        obs, sim = match_nearest(pl_d, pl_a, sim_d, sim_a)
        N = len(obs)
        if N == 0: continue
        mo, ms = sum(obs)/N, sum(sim)/N
        records.append({
            'Reservoir':      lake,
            'Method':         label,
            'N':              N,
            'Planet_mean_ha': round(mo, 1),
            'Method_mean_ha': round(ms, 1),
            'Bias_ha':        round(ms - mo, 1),
            'RMSE_ha':        round(math.sqrt(sum((s-o)**2 for o,s in zip(obs,sim))/N), 1),
            'KGE':            round(kge(obs, sim), 3),
        })

df_table3 = pd.DataFrame(records)
print(df_table3.to_string(index=False))

out_csv = BASE_STAT / 'table3_ablation_metrics.csv'
df_table3.to_csv(out_csv, index=False)
print(f'\nTabela 3 salva em: {out_csv}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

method_order = ['SVM VV+VH (app)', 'SVM VV-only', 'Otsu VV']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('white')

for metric, ax, cmap, vmin, title in [
    ('KGE',     axes[0], 'RdYlGn',   0.0, 'KGE'),
    ('RMSE_ha', axes[1], 'RdYlGn_r', 0.0, 'RMSE (ha)'),
]:
    mat = np.array([[df_table3[(df_table3.Reservoir==l)&(df_table3.Method==m)][metric].values[0]
                     if len(df_table3[(df_table3.Reservoir==l)&(df_table3.Method==m)]) else float('nan')
                     for l in LAKES] for m in method_order], dtype=float)
    vmax = 1.0 if metric == 'KGE' else np.nanmax(mat)*1.05
    im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.85)
    ax.set_xticks(range(len(LAKES)));  ax.set_xticklabels(LAKES, fontsize=10)
    ax.set_yticks(range(len(method_order))); ax.set_yticklabels(method_order, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    for i, m in enumerate(method_order):
        for j, l in enumerate(LAKES):
            v = mat[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}' if metric=='KGE' else f'{v:.1f}',
                        ha='center', va='center', fontsize=11, fontweight='bold')

plt.suptitle('Ablation Study — Method Comparison vs. PlanetScope',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig_out = Path('../validation_data/ablation_metrics_heatmap.png')
plt.savefig(fig_out, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Heatmap salvo: {fig_out}')


---
## Análise 6 — Série temporal comparativa com PlanetScope

Gráfico de 4 painéis com as séries de cada método e os pontos de referência
PlanetScope, com métricas KGE e RMSE anotadas por painel.

In [ ]:
import matplotlib.dates as mdates

LAKE_LABELS = {
    'Rosamarina': 'Invaso Rosamarina', 'Pozzillo': 'Invaso Pozzillo',
    'Poma':       'Invaso Poma',       'Ancipa':   'Invaso Ancipa',
}
PLOT_METHODS = [
    ('app_svm',  None,         None,           'SVM VV+VH (app)', '#1a6faf', '-',  1.8),
    ('ablation', 'SVM_VVonly', 'areaVVonly_ha','SVM VV-only',     '#e07b39', '--', 1.5),
    ('ablation', 'Otsu',       'areaOtsu_ha',  'Otsu VV',         '#2ca02c', ':',  1.5),
]

fig, axes = plt.subplots(4, 1, figsize=(13, 15), sharex=False)
fig.patch.set_facecolor('white')

for ax, lake in zip(axes, LAKES):
    for source, raw_method, raw_col, label, color, ls, lw in PLOT_METHODS:
        if source == 'app_svm':
            d, a = read_csv_dedup(SVM_FILES[lake])
        else:
            rd, ra = read_ablation_raw(lake, raw_method, raw_col)
            d, a   = apply_pipeline(rd, ra)
        ax.plot(d, a, ls, color=color, lw=lw, ms=3, marker='o',
                markerfacecolor=color, markeredgewidth=0, alpha=0.88,
                label=label, zorder=4 if 'VV+VH' in label else 3)

    pl_d, pl_a = read_csv_dedup(PLANET_FILES[lake])
    ax.scatter(pl_d, pl_a, marker='D', s=30, color='#d62728',
               zorder=5, label='PlanetScope', edgecolors='white', lw=0.4)

    sub = df_table3[df_table3.Reservoir == lake]
    parts = [f"{r.Method}: RMSE={r.RMSE_ha:.1f} ha  KGE={r.KGE:.2f}"
             for _, r in sub.iterrows()]
    ax.text(0.01, 0.97, '\n'.join(parts), transform=ax.transAxes,
            fontsize=8, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f5', alpha=0.85))

    ax.set_title(LAKE_LABELS[lake], fontsize=12, fontweight='bold', pad=4)
    ax.set_ylabel('Area (ha)', fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=9)
    ax.tick_params(axis='y', labelsize=9)
    ax.grid(True, alpha=0.25, ls='--')
    ax.set_facecolor('#fafafa')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

handles, lbls = axes[0].get_legend_handles_labels()
fig.legend(handles, lbls, loc='lower center', ncol=4, fontsize=11,
           frameon=True, framealpha=0.9, bbox_to_anchor=(0.5, 0.005))
fig.suptitle(
    'Ablation Study — Water Surface Area (May 2024 - May 2025)\n'
    'SVM VV+VH (app)  vs.  SVM VV-only  vs.  Otsu VV  vs.  PlanetScope',
    fontsize=12, fontweight='bold', y=0.998)
plt.tight_layout(rect=[0, 0.055, 1, 0.998])

fig_out2 = Path('../validation_data/ablation_timeseries_final.png')
plt.savefig(fig_out2, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Figura salva: {fig_out2}')
